In [2]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

In [3]:
import joblib
import pandas as pd

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

from sklearn.model_selection import train_test_split

from src.preprocessing import clean_data

In [4]:
df = pd.read_csv("../data/creditcard.csv")

In [5]:
df = clean_data(df)

In [6]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [7]:
best_pipeline = joblib.load("../models/best_model.pkl")

best_pipeline

,steps,"[('preprocessing', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](30,)","['Time','V1','V2',...,'V27','V28','Amount']"
n_features_in_,int,30
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('amount_log', ...), ('hour_sin', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3


In [8]:
# Keep each forest single-threaded while RandomizedSearchCV parallelizes the CV fits.
best_pipeline.set_params(model__n_jobs=1)

param_distributions = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [5, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

In [9]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=best_pipeline,
    param_distributions=param_distributions,
    n_iter=8,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=3,
    pre_dispatch="2*n_jobs",
    random_state=42
)

search.fit(X_train, y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [5, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",8
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",3
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inch

In [10]:
best_tuned_pipeline = search.best_estimator_

y_pred = best_tuned_pipeline.predict(X_test)
y_proba = best_tuned_pipeline.predict_proba(X_test)[:, 1]

tuned_results = pd.DataFrame([{
    "model": type(best_tuned_pipeline.named_steps["model"]).__name__,
    "best_cv_AUPRC": search.best_score_,
    "test_AUPRC": average_precision_score(y_test, y_proba),
    "test_ROC_AUC": roc_auc_score(y_test, y_proba),
    "test_Precision": precision_score(y_test, y_pred),
    "test_Recall": recall_score(y_test, y_pred),
    "test_F1": f1_score(y_test, y_pred),
    "best_params": search.best_params_
}])

tuned_results

,model,best_cv_AUPRC,test_AUPRC,test_ROC_AUC,test_Precision,test_Recall,test_F1,best_params
0,RandomForestClassifier,0.838828,0.80173,0.977361,0.83908,0.768421,0.802198,"{'model__n_estimators': 200, 'model__min_sampl..."


## Hyperparameter Tuning Results

Following model selection, hyperparameter tuning was performed on the Random Forest classifier using RandomizedSearchCV with 5-fold stratified cross-validation and AUPRC as the optimization metric.

The tuned model achieved a cross-validation AUPRC of 0.839 and a test AUPRC of 0.802. These results are very close to those obtained with the baseline Random Forest model, indicating that the initial parameter configuration was already near-optimal for this dataset.

Compared to the untuned model, the tuned version achieved a slight increase in recall (76.8% versus 75.8%), allowing the model to detect a slightly higher proportion of fraudulent transactions. However, this improvement came at the cost of reduced precision (83.9% versus 93.5%), meaning that more legitimate transactions were incorrectly flagged as fraudulent.

Overall, hyperparameter tuning did not lead to a significant improvement in predictive performance. The relatively small difference between the baseline and tuned models suggests that Random Forest is robust on this dataset and that model performance is influenced more by the available features and class imbalance than by parameter optimization alone.

Consequently, the tuned model will be retained for threshold optimization, where the decision threshold can be adjusted to achieve a more suitable balance between fraud detection rate and false positive rate according to business requirements.

In [11]:
joblib.dump(best_tuned_pipeline, "../models/best_tuned_model.pkl")

tuned_results.to_csv("../models/best_tuned_model_results.csv", index=False)